# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [3]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [4]:
# TODO: Load environment variables
load_dotenv()

True

### VectorDB Instance

In [5]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
chroma_client = chromadb.PersistentClient(path="chromadb")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


### Collection

In [6]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=os.getenv("OPENAI_BASE_URL"),
)

In [7]:
# TODO: Create a collection
# Choose any name you want
collection = chroma_client.get_or_create_collection(
name="udaplay",
embedding_function=embedding_fn
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


### Add documents

In [8]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


### Test semantic search

In [9]:
queries = [
    "When was Pokémon Gold and Silver released?",
    "first 3D platformer Mario game",
    "PlayStation 5 games",
]

for q in queries:
    results = collection.query(query_texts=[q], n_results=3)
    print(f"\nQuery: {q}")
    for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        print(f"  - {meta['Name']} ({meta['Platform']}, {meta['YearOfRelease']})  distance={dist:.3f}")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



Query: When was Pokémon Gold and Silver released?
  - Pokémon Gold and Silver (Game Boy Color, 1999)  distance=0.226
  - Pokémon Ruby and Sapphire (Game Boy Advance, 2002)  distance=0.293
  - Super Mario 64 (Nintendo 64, 1996)  distance=0.441

Query: first 3D platformer Mario game
  - Super Mario 64 (Nintendo 64, 1996)  distance=0.207
  - Super Mario World (Super Nintendo Entertainment System (SNES), 1990)  distance=0.256
  - Mario Kart 8 Deluxe (Nintendo Switch, 2017)  distance=0.376

Query: PlayStation 5 games
  - Gran Turismo 5 (PlayStation 3, 2010)  distance=0.345
  - Marvel's Spider-Man 2 (PlayStation 5, 2023)  distance=0.359
  - Marvel's Spider-Man (PlayStation 4, 2018)  distance=0.384
